# 02 Data Skew

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Detect data skew (a few keys holding most of the rows) and fix it with the right technique: AQE skew join handling, broadcasting, processing hot keys separately, <b>salting</b> for joins and aggregations, and two-phase aggregation.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"One task takes much longer than the rest. Why?" is a classic interview scenario, and skew is the answer most of the time. Skew wastes a cluster: 199 tasks finish in a minute and one runs for an hour, so the stage takes an hour. It also causes spill and out-of-memory errors in the overloaded task.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Why skew hurts</b><br>
Wide operations send all rows with the same key to the same partition. If one key has 40% of the rows, one task gets 40% of the work:<br><code>`</code><br>key distribution:  IN ████████████████████ 40%   other 199 keys: ▏▏▏▏▏...<br>tasks after shuffle:  task 7: 40% of data (long)   tasks 0-199: tiny (done quickly, then idle)<br><code>`</code><br><b>Detect:</b> key frequencies (<code>groupBy(key).count()</code>), partition sizes after a shuffle, task duration max ≫ median in the profile or Spark UI.<br><br>
<b>Fix (in order of preference):</b><br>| Technique | Works for | Idea |<br>|---|---|---|<br>| AQE skew join | Sort-merge joins | Spark splits the skewed partition automatically (on by default) |<br>| Broadcast the other side | Joins with a small side | No shuffle of the big side, so no skew problem |<br>| Filter / handle hot keys separately | Joins and aggregations | Process the few hot keys differently, union the results |<br>| Salting | Joins and aggregations | Add a random suffix to spread a hot key across N partitions |<br>| Two-phase aggregation | Aggregations | Aggregate by (key, salt) first, then by key |
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A marketplace's order table had one seller, the platform's own store, with 35% of all orders. The nightly join to seller details and the aggregation per seller each had one task running 50 minutes. Broadcasting the 30,000-row seller table fixed the join. Two-phase aggregation with 16 salt values fixed the aggregation. The job went from 75 minutes to 9.
</div>

## Setup: skewed data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates 2 million orders where one seller (<code>S0</code>) has about 60% of the rows, and a sellers dimension.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The top keys by row count: <code>S0</code> with about 1.2 million rows, every other seller with a few hundred.
</div>

In [0]:
import time
from pyspark.sql import functions as F

orders = spark.range(2_000_000).select(
    F.col("id").alias("order_id"),
    F.when(F.rand(seed=7) < 0.6, F.lit("S0"))
     .otherwise(F.concat(F.lit("S"), ((F.rand(seed=8) * 2000).cast("int") + 1).cast("string"))).alias("seller_id"),
    F.round(F.rand(seed=9) * 200, 2).alias("amount"),
)
sellers = spark.range(2001).select(F.concat(F.lit("S"), F.col("id").cast("string")).alias("seller_id"),
                                   (F.col("id") % 10).alias("region_id"))

orders.groupBy("seller_id").count().orderBy(F.desc("count")).show(5)

def timed(label, df):
    start = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label:<45} {time.time() - start:6.2f}s")

def partition_sizes(df, label):
    sizes = sorted((r["count"] for r in df.groupBy(F.spark_partition_id()).count().collect()), reverse=True)
    print(f"{label:<45} partitions: {len(sizes):>3} | biggest: {sizes[0]:>8} | median: {sizes[len(sizes) // 2]:>7}")

## 1. Detect skew

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Measures skew three ways: the share of the top key, the partition sizes after a hash shuffle by that key, and a skew ratio.<br><br>
<b>Why it matters</b><br>Before fixing skew, prove it, and find the hot keys. A simple rule: if the biggest partition is many times the median, you have skew.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>S0</code> holds about 60% of the rows. After <code>repartition(8, "seller_id")</code>, one partition is far bigger than the median, with a skew ratio much larger than 1.
</div>

In [0]:
total = orders.count()
top = orders.groupBy("seller_id").count().orderBy(F.desc("count")).first()
print(f"hot key {top['seller_id']}: {top['count'] / total:.0%} of all rows")

partition_sizes(orders.repartition(8, "seller_id"), "hash-partitioned by seller_id")

stats = orders.groupBy("seller_id").count().agg(F.max("count").alias("mx"), F.expr("percentile_approx(count, 0.5)").alias("med")).first()
print(f"skew ratio (max key / median key): {stats['mx'] / stats['med']:.0f}x")

## 2. Skewed join: AQE and broadcast

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins orders to sellers with a forced sort-merge join, first with AQE skew handling as configured, then with a broadcast join.<br><br>
<b>Why it matters</b><br>With a sort-merge join, the <code>S0</code> partition is huge. AQE's skew join splits it (when thresholds are met, see lesson 02_13). But the simplest fix here is to <b>broadcast</b> the small sellers table, which removes the shuffle of the orders entirely, and with it the skew.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The broadcast join is the fastest and produces the same totals.
</div>

In [0]:
smj = orders.join(sellers.hint("merge"), "seller_id").groupBy("region_id").agg(F.sum("amount").alias("total"))
bhj = orders.join(F.broadcast(sellers), "seller_id").groupBy("region_id").agg(F.sum("amount").alias("total"))
timed("sort-merge join (skewed partition)", smj)
timed("broadcast join (no shuffle of orders)", bhj)
print("same result:", sorted((r[0], round(r[1], 2)) for r in smj.collect()) == sorted((r[0], round(r[1], 2)) for r in bhj.collect()))

## 3. Salting a join (when both sides are big)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Spreads the hot key over N "salt" buckets: adds a random salt (0..N-1) to the big side, replicates the matching rows of the other side once per salt, and joins on <code>(key, salt)</code>.<br><br>
<b>Why it matters</b><br>When neither side can be broadcast, salting turns one giant partition into N normal ones. The cost is replicating the other side's rows N times, so salt only the hot keys if possible.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The salted key's partitions are much more balanced than the unsalted ones, and the join result is identical.
</div>

In [0]:
N = 16
orders_salted = orders.withColumn("salt", (F.rand(seed=11) * N).cast("int"))
sellers_salted = sellers.crossJoin(spark.range(N).withColumnRenamed("id", "salt").withColumn("salt", F.col("salt").cast("int")))

partition_sizes(orders.repartition(32, "seller_id"), "unsalted: by seller_id")
partition_sizes(orders_salted.repartition(32, "seller_id", "salt"), "salted: by (seller_id, salt)")

salted_join = (orders_salted.join(sellers_salted.hint("merge"), ["seller_id", "salt"])
               .groupBy("region_id").agg(F.sum("amount").alias("total")))
timed("salted sort-merge join", salted_join)
print("same result:", sorted((r[0], round(r[1], 2)) for r in salted_join.collect()) == sorted((r[0], round(r[1], 2)) for r in bhj.collect()))

## 4. Skewed aggregation: two-phase (salted) aggregation

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes distinct customers per seller-like key with a heavy aggregation, first directly, then in two phases: aggregate by <code>(key, salt)</code>, then combine by key.<br><br>
<b>Why it matters</b><br>Simple aggregates like <code>sum</code> and <code>count</code> are partially aggregated before the shuffle, so they rarely suffer from skew. Aggregates that keep a lot of state per key, such as <code>collect_list</code>, <code>collect_set</code> or exact <code>countDistinct</code>, put all of the hot key's values into one task. Two-phase aggregation splits that work.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Both versions give the same number of distinct amounts for <code>S0</code>. On this small dataset the two-phase version may even be slower, because it adds a second shuffle. Its benefit appears when the hot key's state is too large for one task: the direct version then spills or runs out of memory in a single straggler task, while the two-phase version spreads that work over N tasks.
</div>

In [0]:
direct = orders.groupBy("seller_id").agg(F.size(F.collect_set("amount")).alias("distinct_amounts"))

two_phase = (orders.withColumn("salt", (F.rand(seed=12) * N).cast("int"))
             .groupBy("seller_id", "salt").agg(F.collect_set("amount").alias("amounts"))          # phase 1: spread the hot key
             .groupBy("seller_id").agg(F.size(F.array_distinct(F.flatten(F.collect_list("amounts")))).alias("distinct_amounts")))  # phase 2

timed("direct collect_set per seller", direct)
timed("two-phase (salted) aggregation", two_phase)
print("S0 direct   :", direct.filter("seller_id = 'S0'").first()["distinct_amounts"])
print("S0 two-phase:", two_phase.filter("seller_id = 'S0'").first()["distinct_amounts"])

## 5. Handle the hot key separately

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Splits the data into the hot key and the rest, processes each with the best strategy, and unions the results.<br><br>
<b>Why it matters</b><br>Often only one or two keys are hot, and they're known (a default value, a test account, a marketplace's own store). Treating them separately keeps the main path simple. It also surfaces a frequent root cause: hot keys are often <b>bad data</b>, such as <code>NULL</code>, <code>'unknown'</code> or <code>-1</code>, which should be filtered or fixed rather than optimized.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Totals per region identical to the broadcast join result.
</div>

In [0]:
hot = ["S0"]
hot_part = orders.filter(F.col("seller_id").isin(hot)).join(F.broadcast(sellers.filter(F.col("seller_id").isin(hot))), "seller_id")
rest_part = orders.filter(~F.col("seller_id").isin(hot)).join(sellers, "seller_id")
combined = hot_part.unionByName(rest_part).groupBy("region_id").agg(F.sum("amount").alias("total"))
print("same result:", sorted((r[0], round(r[1], 2)) for r in combined.collect()) == sorted((r[0], round(r[1], 2)) for r in bhj.collect()))

## Under the hood

```
shuffle by key:  partition = hash(key) % N
   hot key K (60% of rows) -> ALL its rows to partition hash(K) % N -> one huge task

AQE skew join:   after the map stage, partitions > 5x median and > 256 MB are split into
                 sub-partitions; the matching partition of the other side is read once per split
salting:         partition = hash(key, salt) % N  -> K's rows spread over up to N partitions
broadcast:       no shuffle of the big side at all
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> skew only exists after a shuffle: it's a property of how keys map to partitions.

Look for skew in the profile or Spark UI as **max ≫ median** for task duration, shuffle read and spill within one stage.

In [0]:
salted_join.explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: adding more executors doesn't help</b><br>
One task is doing the work. More cores don't split a single partition. Fix the skew.<br><br>
<b>⛔ Problem: AQE didn't split the skewed partition</b><br>The partition is under the thresholds (256 MB and 5× median by default), or it's an aggregation, not a join. Use salting or two-phase aggregation.<br><br>
<b>⛔ Problem: salting made the job slower</b><br>The other side was replicated N times. Salt only the hot keys, or choose a smaller N.<br><br>
**⛔ Problem: the hot key is <code>NULL</code> or a placeholder**<br>That's often a data quality bug. Filter or fix those rows instead of optimizing for them.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. One task takes much longer than the others. Why, and what do you do?</b><br>
Usually data skew: after a shuffle, one partition holds far more data because a key is very frequent. I confirm it with the task time distribution and key counts. Then I fix it: let AQE's skew join handle it, broadcast the smaller side, process the hot keys separately, or salt the key so the hot key spreads over several partitions.<br><br>

<b>🎤 2. What is salting?</b><br>
Adding a random number (0 to N-1) to a skewed key on the big side, and replicating the other side's rows for each salt value, so the hot key is split across N partitions. For aggregations, aggregate by key and salt first, then by key.<br><br>

<b>🎤 3. Why don't simple sums suffer much from skew?</b><br>
Spark does partial aggregation in each map task, so only one row per key per task is shuffled. Aggregates that keep all values, such as <code>collect_list</code> or exact distinct counts, can't shrink like that and do suffer.<br><br>

<b>🎤 4. How does AQE handle skew?</b><br>
After the map stage it knows the partition sizes. In sort-merge joins it splits partitions larger than a factor of the median and above a size threshold into smaller ones, and reads the matching data from the other side for each split.<br><br>

<b>🎤 5. How do you find skewed keys?</b><br>
<code>groupBy(key).count().orderBy(desc)</code> on the join or grouping key, partition sizes after a shuffle, and the task metrics in the profile or Spark UI.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. In a join stage, one task processes 30 GB while the other 199 process about 150 MB each. Which Spark feature can automatically split that partition?</b><br>
A. Delta data skipping<br>
B. Adaptive Query Execution skew join optimization<br>
C. Column pruning<br>
D. Liquid clustering<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A large fact table is joined to a 20 MB dimension, and the join key is heavily skewed. What is the simplest fix?</b><br>
A. Salt both tables with 1,000 salt values<br>
B. Broadcast the dimension table<br>
C. Increase executor memory only<br>
D. Sort the fact table first<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Broadcasting avoids shuffling the skewed side at all.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a skewed dataset where 3 keys hold 70% of the rows, and show the partition sizes after <code>repartition(16, key)</code></li><li>Fix a sort-merge join on that key with salting, salting only the 3 hot keys</li><li>Compute <code>countDistinct</code> of a value column per key, both directly and with a two-phase approach, and check that the results match</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Three hot keys
r = F.rand(seed=21)
data = spark.range(1_000_000).select(
    F.when(r < 0.3, "K1").when(r < 0.55, "K2").when(r < 0.7, "K3")
     .otherwise(F.concat(F.lit("K"), ((F.rand(seed=22) * 500).cast("int") + 10).cast("string"))).alias("key"),
    (F.rand(seed=23) * 10_000).cast("int").alias("v"))
partition_sizes(data.repartition(16, "key"), "by key")

# 2. Salt only the hot keys (salt 0 for the rest), replicate only the hot keys on the other side
HOT, N2 = ["K1", "K2", "K3"], 8
dim = data.select("key").distinct().withColumn("label", F.concat(F.lit("label_"), F.col("key")))
data_s = data.withColumn("salt", F.when(F.col("key").isin(HOT), (F.rand(seed=24) * N2).cast("int")).otherwise(0))
dim_s = (dim.filter(F.col("key").isin(HOT)).crossJoin(spark.range(N2).select(F.col("id").cast("int").alias("salt")))
         .unionByName(dim.filter(~F.col("key").isin(HOT)).withColumn("salt", F.lit(0))))
partition_sizes(data_s.repartition(16, "key", "salt"), "by (key, salt)")
print("rows joined:", data_s.join(dim_s.hint("merge"), ["key", "salt"]).count(), "of", data.count())

# 3. Exact distinct count, direct vs two-phase
direct = data.groupBy("key").agg(F.countDistinct("v").alias("n"))
two = (data.withColumn("salt", (F.rand(seed=25) * N2).cast("int"))
       .groupBy("key", "salt").agg(F.collect_set("v").alias("vs"))
       .groupBy("key").agg(F.size(F.array_distinct(F.flatten(F.collect_list("vs")))).alias("n")))
print("results match:", sorted(direct.collect()) == sorted(two.collect()))

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Skew = a few keys hold most rows, so one task does most of the work after a shuffle</li><li>Detect: key counts, partition sizes, task max ≫ median</li><li>Fix: AQE skew join (automatic, joins), broadcast the small side, isolate hot keys, salting, two-phase aggregation</li><li>Simple sums and counts are protected by partial aggregation. <code>collect_*</code> and exact distinct aren't</li><li>Hot keys are often bad data (<code>NULL</code>, <code>unknown</code>). Fix the data first</li></ul>
</div>

| Task | Code |
|---|---|
| Find hot keys | `df.groupBy("k").count().orderBy(F.desc("count"))` |
| Partition sizes | `df.repartition(n, "k").groupBy(F.spark_partition_id()).count()` |
| Broadcast | `big.join(F.broadcast(small), "k")` |
| Salt | `.withColumn("salt", (F.rand() * N).cast("int"))` + replicate the other side with `crossJoin(spark.range(N))` |
| Two-phase agg | `groupBy("k", "salt").agg(...)`, then `groupBy("k").agg(...)` |

## Git commit

```
git add 05_optimization/02_data_skew.ipynb
git commit -m "add 05_02 data skew notebook"
```